Data Science Systems Capstone - Adventure Works 

This code is used to import required libraries. 

In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

/opt/homebrew/opt/apache-spark/libexec


This code is used to instantiate global variables. 

In [2]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "host_name": "localhost",
    "port": "3306",
    "db_name": "adventureworks",
    "conn_props": {
        "user": "root",
        "password": "Marcel1n$",
        "driver": "com.mysql.cj.jdbc.Driver"
    }
}

# --------------------------------------------------------------------------------
# Specify MongoDB Cluster Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "cluster_location" : "atlas",
    "user_name" : "jemarcelin14_db_user",
    "password" : "6iWDKBwHzlNtj7Ok",
    "cluster_name" : "cluster1001",
    "cluster_subnet" : "qnfuayc",
    "db_name" : "adventureworks",
    "collection" : "",
    "null_column_threshold" : 0.5
}
# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.path.expanduser("~"), 'lab_data')
data_dir = os.path.join(base_dir, 'adventureworks')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

sales_orders_stream_dir = os.path.join(stream_dir, 'sales_orders')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

sales_orders_output_bronze = os.path.join(database_dir, 'fact_sales_orders', 'bronze')
sales_orders_output_silver = os.path.join(database_dir, 'fact_sales_orders', 'silver')
sales_orders_output_gold   = os.path.join(database_dir, 'fact_sales_orders', 'gold')

Define the functions that will be used later in the code. 

In [3]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []

    '''Fetch each item in the directory, and filter out any directories.'''
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    '''Populate lists with the Size and Last Modification DateTime for each file in the directory.'''
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
        
    print(f"The stream has processed {len(query.recentProgress)} batchs")


def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"
        

def drop_null_columns(df, threshold):
    '''Drop Columns having a percentage of NULL values that exceeds the given 'threshold' parameter value.'''
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    
    return df_dropped
    
    
def get_mysql_dataframe(spark_session, sql_query : str, **args):
    '''Create a JDBC URL to the MySQL Database'''
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    
    '''Invoke the spark.read.format("jdbc") function to query the database, and fill a DataFrame.'''
    dframe = spark_session.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", args['conn_props']['driver']) \
    .option("user", args['conn_props']['user']) \
    .option("password", args['conn_props']['password']) \
    .option("query", sql_query) \
    .load()
    
    return dframe
    

def get_mongo_uri(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
        
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"

    return uri


def get_spark_conf_args(spark_jars : list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    
    sparkConf_args = {
        "app_name" : "PySpark Northwind Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[0:-2],
        "database_dir" : sql_warehouse_dir
    }
    
    return sparkConf_args
    

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name']) \
    .setMaster(args['worker_threads']) \
    .set('spark.driver.memory', '4g') \
    .set('spark.executor.memory', '2g') \
    .set('spark.jars', args['spark_jars']) \
    .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.13:11.0.1') \
    .set('spark.mongodb.read.connection.uri', args['mongo_uri']) \
    .set('spark.mongodb.write.connection.uri', args['mongo_uri']) \
    .set('spark.sql.adaptive.enabled', 'false') \
    .set('spark.sql.debug.maxToStringFields', 35) \
    .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
    .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .set('spark.sql.streaming.schemaInference', 'true') \
    .set('spark.sql.warehouse.dir', args['database_dir']) \
    .set('spark.streaming.stopGracefullyOnShutdown', 'true')

    return sparkConf

def get_mongo_client(**args):
    '''Get MongoDB Client Connection'''
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())

    elif args['cluster_location'] == "local":
        client = pymongo.MongoClient(mongo_uri)
        
    else:
        raise Exception("A MongoDB Client could not be created.")

    return client
    
    
# TODO: Rewrite this to leverage PySpark?
def set_mongo_collections(mongo_client, db_name : str, data_directory : str, json_files : list):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()
    

def get_mongodb_dataframe(spark_session, **args):
    '''Query MongoDB, and create a DataFrame'''
    dframe = spark_session.read.format("mongodb") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']) \
        .load()

    '''Drop the '_id' index column to clean up the response.'''
    dframe = dframe.drop('_id')
    
    '''Call the drop_null_columns() function passing in the dataframe.'''
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    
    return dframe

Delete the existing data lakehouse folders to ensure that it runs smoothly (and can be rerun). 

In [4]:
remove_directory_tree(database_dir)

"Directory '/Users/jericmarcelin/Desktop/spark-warehouse/adventureworks_dlh.db' has been removed successfully."

In [5]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"

jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")

jars.append(mysql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)
sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/08 14:43:10 WARN Utils: Your hostname, Jerics-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.82 instead (on interface en0)
26/05/08 14:43:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/jericmarcelin/.ivy2.5.2/cache
The jars for the packages stored in: /Users/jericmarcelin/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-21dbaf95-161d-4065-93f1-76f1b678dfd0;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;11.0.1 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mo

Create a new metadata database for AdventureWorks Data Lakehouse

In [6]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Data Project 2 - AdventureWorks Data Lakehouse'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Capstone Project');
"""
spark.sql(sql_create_db)

DataFrame[]

Verify the location of the source data for this project. 

In [7]:
get_file_info(batch_dir)


,name,size,modification_time
0,.DS_Store,6148,2026-04-27 17:53:29.797371387
1,dim_employee.csv,53190,2026-04-27 17:08:52.264521836
2,dim_product.json,218523,2026-04-27 17:15:48.764597654
3,dim_productcategory.json,5077,2026-04-27 17:31:54.409333944
4,dim_salesterritory.csv,672,2026-04-27 17:34:00.816847086


Populate the employee dimension. 

In [8]:
employee_csv = os.path.join(batch_dir, 'dim_employee.csv')
print(employee_csv)

df_dim_employees = spark.read.format('csv').options(header='true', inferSchema='true').load(employee_csv)
df_dim_employees.toPandas().head(2)

/Users/jericmarcelin/lab_data/adventureworks/batch/dim_employee.csv


,EmployeeID,NationalIDNumber,LoginID,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,14417807,adventure-works\guy1,16,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15,M,M,1996-07-31,0,21,30,1
1,2,253022876,adventure-works\kevin0,6,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03,S,M,1997-02-26,0,42,41,1


Make necessary transformations to clean data for the employees dimension. 

In [9]:
# ----------------------------------------------------------------------------------
# Rename the 'id' column to 'employee_id' ------------------------------------------
# ----------------------------------------------------------------------------------
df_dim_employees = df_dim_employees.withColumnRenamed("id", "employee_id")

# ----------------------------------------------------------------------------------
# Add Primary Key column using SQL Windowing function: ROW_NUMBER()
# ----------------------------------------------------------------------------------
df_dim_employees.createOrReplaceTempView("employees")
sql_employees = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY EmployeeID) AS employee_key
    FROM employees;
"""
df_dim_employees = spark.sql(sql_employees)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['employee_key', 'EmployeeID', 'FirstName', 'MiddleName', 'LastName',
                   'Title', 'EmailAddress', 'Phone', 'BirthDate', 'MaritalStatus',
                   'Gender', 'HireDate', 'SalariedFlag', 'VacationHours',
                   'SickLeaveHours', 'CurrentFlag']

df_dim_employees = df_dim_employees[ordered_columns]
df_dim_employees.toPandas().head(2)

,employee_key,EmployeeID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,1,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15,M,M,1996-07-31,0,21,30,1
1,2,2,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03,S,M,1997-02-26,0,42,41,1


The employees dimension is saved into the Data Lakehouse. 

In [10]:
df_dim_employees.write.saveAsTable(f"{dest_database}.dim_employee", mode="overwrite")

Tests and shows the employee dimension after importing and altering. 

In [11]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_employee;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_employee LIMIT 2").toPandas()

+--------------------+------------------+-------+
|            col_name|         data_type|comment|
+--------------------+------------------+-------+
|        employee_key|               int|   NULL|
|          EmployeeID|               int|   NULL|
|           FirstName|            string|   NULL|
|          MiddleName|            string|   NULL|
|            LastName|            string|   NULL|
|               Title|            string|   NULL|
|        EmailAddress|            string|   NULL|
|               Phone|            string|   NULL|
|           BirthDate|         timestamp|   NULL|
|       MaritalStatus|            string|   NULL|
|              Gender|            string|   NULL|
|            HireDate|         timestamp|   NULL|
|        SalariedFlag|               int|   NULL|
|       VacationHours|               int|   NULL|
|      SickLeaveHours|               int|   NULL|
|         CurrentFlag|               int|   NULL|
|                    |                  |       |


,employee_key,EmployeeID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,1,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15,M,M,1996-07-31,0,21,30,1
1,2,2,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03,S,M,1997-02-26,0,42,41,1


Populate the Sales Territory Dimension

In [12]:
salesterritory_csv = os.path.join(batch_dir, 'dim_salesterritory.csv')
print(salesterritory_csv)

df_dim_salesterritory = spark.read.format('csv').options(header='true', inferSchema='true').load(salesterritory_csv)
df_dim_salesterritory.toPandas().head(2)

/Users/jericmarcelin/lab_data/adventureworks/batch/dim_salesterritory.csv


,TerritoryID,TerritoryName,CountryRegionCode,TerritoryGroup,SalesYTD,SalesLastYear,CostYTD,CostLastYear
0,1,Northwest,US,North America,5.767342e+06,3.298694e+06,0,0
1,2,Northeast,US,North America,3.857164e+06,3.607149e+06,0,0


Make necessary transformations to clean data for the sales territory dimension. 

In [13]:
# ----------------------------------------------------------------------------------
# Rename TerritoryID to territory_id if present
# ----------------------------------------------------------------------------------
if 'TerritoryID' in df_dim_salesterritory.columns:
    df_dim_salesterritory = df_dim_salesterritory.withColumnRenamed("TerritoryID", "territory_id")

# ----------------------------------------------------------------------------------
# Add Primary Key column using SQL Windowing function: ROW_NUMBER()
# ----------------------------------------------------------------------------------
df_dim_salesterritory.createOrReplaceTempView("salesterritory")
sql_salesterritory = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY territory_id) AS territory_key
    FROM salesterritory;
"""
df_dim_salesterritory = spark.sql(sql_salesterritory)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['territory_key', 'territory_id', 'TerritoryName',
                   'CountryRegionCode', 'TerritoryGroup',
                   'SalesYTD', 'SalesLastYear', 'CostYTD', 'CostLastYear']

df_dim_salesterritory = df_dim_salesterritory[ordered_columns]
df_dim_salesterritory.toPandas().head(2)

,territory_key,territory_id,TerritoryName,CountryRegionCode,TerritoryGroup,SalesYTD,SalesLastYear,CostYTD,CostLastYear
0,1,1,Northwest,US,North America,5.767342e+06,3.298694e+06,0,0
1,2,2,Northeast,US,North America,3.857164e+06,3.607149e+06,0,0


Save dim_salesterritory to the Data Lakehouse

In [14]:

df_dim_salesterritory.write.saveAsTable(f"{dest_database}.dim_salesterritory", mode="overwrite")

Tests and shows the sales territory dimenson after importing and altering. 

In [15]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_salesterritory;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_salesterritory LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|       territory_key|                 int|   NULL|
|        territory_id|                 int|   NULL|
|       TerritoryName|              string|   NULL|
|   CountryRegionCode|              string|   NULL|
|      TerritoryGroup|              string|   NULL|
|            SalesYTD|              double|   NULL|
|       SalesLastYear|              double|   NULL|
|             CostYTD|                 int|   NULL|
|        CostLastYear|                 int|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|  dim_salesterritory|       |
|        Created Time|Fri May 08 14:43:...|       |
|         Last Access|             UNKNOWN|       |
|          C

,territory_key,territory_id,TerritoryName,CountryRegionCode,TerritoryGroup,SalesYTD,SalesLastYear,CostYTD,CostLastYear
0,1,1,Northwest,US,North America,5.767342e+06,3.298694e+06,0,0
1,2,2,Northeast,US,North America,3.857164e+06,3.607149e+06,0,0


Fetch Data from MongoDB atlas DB

This code creates a New MongoDB Database, and Load the product JSON File and product category JSON into a New MongoDB Collection.

In [16]:
client = get_mongo_client(**mongodb_args)

json_files = {"dim_product" : "dim_product.json",
              "dim_productcategory" : "dim_productcategory.json"
             }

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files)

Fetch the data from dim_product. 

In [17]:
mongodb_args["collection"] = "dim_product"

df_dim_product = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_product.toPandas().head(2)

,Color,ListPrice,ProductCategory,ProductID,ProductLine,ProductModel,ProductName,ProductNumber,ProductSubcategory,SellStartDate,StandardCost
0,None,0.0,None,1,None,None,Adjustable Race,AR-5381,None,1998-06-01 00:00:00,0.0
1,None,0.0,None,2,None,None,Bearing Ball,BA-8327,None,1998-06-01 00:00:00,0.0


Make Necessary Transformations to the New Dataframe and drop columns with same names. 

In [18]:
# ----------------------------------------------------------------------------------
# Rename ProductID to product_id and drop if product_key already exists
# ----------------------------------------------------------------------------------
df_dim_product = df_dim_product.withColumnRenamed("ProductID", "product_id")

if 'product_key' in df_dim_product.columns:
    df_dim_product = df_dim_product.drop('product_key')

# ----------------------------------------------------------------------------------
# Add Primary Key column using SQL Windowing function: ROW_NUMBER()
# ----------------------------------------------------------------------------------
df_dim_product.createOrReplaceTempView("products")
sql_products = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY product_id) AS product_key
    FROM products;
"""
df_dim_product = spark.sql(sql_products)

# ----------------------------------------------------------------------------------
# Reorder Columns — only include columns that exist in the DataFrame
# ----------------------------------------------------------------------------------
all_desired_columns = ['product_key', 'product_id', 'ProductName', 'ProductNumber',
                       'Color', 'StandardCost', 'ListPrice', 'ProductLine',
                       'ProductCategory', 'ProductSubcategory', 'ProductModel',
                       'SellStartDate', 'SellEndDate']

ordered_columns = [c for c in all_desired_columns if c in df_dim_product.columns]
df_dim_product = df_dim_product[ordered_columns]
df_dim_product.toPandas().head(2)

,product_key,product_id,ProductName,ProductNumber,Color,StandardCost,ListPrice,ProductLine,ProductCategory,ProductSubcategory,ProductModel,SellStartDate
0,1,1,Adjustable Race,AR-5381,None,0.0,0.0,None,None,None,None,1998-06-01 00:00:00
1,2,2,Bearing Ball,BA-8327,None,0.0,0.0,None,None,None,None,1998-06-01 00:00:00


Save the new product dimension after it was transformed. 

In [19]:
df_dim_product.write.saveAsTable(f"{dest_database}.dim_product", mode="overwrite")

Test the product dimension. 

In [20]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_product;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_product LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|         product_key|                 int|   NULL|
|          product_id|                 int|   NULL|
|         ProductName|              string|   NULL|
|       ProductNumber|              string|   NULL|
|               Color|              string|   NULL|
|        StandardCost|              double|   NULL|
|           ListPrice|              double|   NULL|
|         ProductLine|              string|   NULL|
|     ProductCategory|              string|   NULL|
|  ProductSubcategory|              string|   NULL|
|        ProductModel|              string|   NULL|
|       SellStartDate|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|           

,product_key,product_id,ProductName,ProductNumber,Color,StandardCost,ListPrice,ProductLine,ProductCategory,ProductSubcategory,ProductModel,SellStartDate
0,1,1,Adjustable Race,AR-5381,None,0.0,0.0,None,None,None,None,1998-06-01 00:00:00
1,2,2,Bearing Ball,BA-8327,None,0.0,0.0,None,None,None,None,1998-06-01 00:00:00


This code is used to populate the product category dimension and fetch the data using Mongo. 

In [21]:
mongodb_args["collection"] = "dim_productcategory"

df_dim_productcategory = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_productcategory.toPandas().head(2)

,ProductCategory,ProductCategoryID,ProductSubcategory,ProductSubcategoryID
0,Bikes,1,Mountain Bikes,1
1,Bikes,1,Road Bikes,2


This is used to create transformations to product category dataframe. 

In [22]:
# ----------------------------------------------------------------------------------
# Rename the 'ProductCategoryID' column to 'productcategory_id'
# ----------------------------------------------------------------------------------
df_dim_productcategory = df_dim_productcategory.withColumnRenamed("ProductCategoryID", "productcategory_id")

# ----------------------------------------------------------------------------------
# Add Primary Key column using SQL Windowing function: ROW_NUMBER()
# ----------------------------------------------------------------------------------
df_dim_productcategory.createOrReplaceTempView("productcategory")
sql_productcategory = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY productcategory_id) AS productcategory_key
    FROM productcategory;
"""
df_dim_productcategory = spark.sql(sql_productcategory)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['productcategory_key', 'productcategory_id', 'ProductCategory',
                   'ProductSubcategoryID', 'ProductSubcategory']

df_dim_productcategory = df_dim_productcategory[ordered_columns]
df_dim_productcategory.toPandas().head(2)

,productcategory_key,productcategory_id,ProductCategory,ProductSubcategoryID,ProductSubcategory
0,1,1,Bikes,1,Mountain Bikes
1,2,1,Bikes,2,Road Bikes


In [23]:
df_dim_productcategory.write.saveAsTable(f"{dest_database}.dim_productcategory", mode="overwrite")

This is used to test and query the product category dimension. 

In [24]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_productcategory;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_productcategory LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
| productcategory_key|                 int|   NULL|
|  productcategory_id|                 int|   NULL|
|     ProductCategory|              string|   NULL|
|ProductSubcategoryID|                 int|   NULL|
|  ProductSubcategory|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table| dim_productcategory|       |
|        Created Time|Fri May 08 14:44:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 4.1.1|       |
|                Type|             MANAGED|       |
|            Provider|             parquet|       |
|            Location|file:/Users/jeric...|       |
+-----------

,productcategory_key,productcategory_id,ProductCategory,ProductSubcategoryID,ProductSubcategory
0,1,1,Bikes,1,Mountain Bikes
1,2,1,Bikes,2,Road Bikes


This code is used to populate the date dimension and fetch the data from the table in MySQl. 
The date dimension code is run in SQL locally prior. 

In [25]:
sql_dim_date = "SELECT * FROM adventureworks.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)
df_dim_date.toPandas().head(2)

,DateKey,Date,Day,DaySuffix,Weekday,WeekDayName,WeekDayName_Short,DOWInMonth,DayOfYear,WeekOfMonth,...,IsWeekend,IsHoliday,HolidayName,SpecialDays,FirstDateofYear,LastDateofYear,FirstDateofMonth,LastDateofMonth,FirstDateofWeek,LastDateofWeek
0,20000101,2000-01-01,1,st,7,Saturday,SAT,1,1,1,...,True,True,New Years Day,None,2000-01-01,2000-12-31,2000-01-01,2000-01-31,1999-12-26,2000-01-01
1,20000102,2000-01-02,2,nd,1,Sunday,SUN,2,2,1,...,True,False,None,None,2000-01-01,2000-12-31,2000-01-01,2000-01-31,2000-01-02,2000-01-08


Save the date dimension in the data lakehouse. 

In [27]:
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

This code is used to test and show the new table that was created. 

In [28]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

+-----------------+-----------+-------+
|         col_name|  data_type|comment|
+-----------------+-----------+-------+
|          DateKey|        int|   NULL|
|             Date|       date|   NULL|
|              Day|    tinyint|   NULL|
|        DaySuffix|    char(2)|   NULL|
|          Weekday|    tinyint|   NULL|
|      WeekDayName|varchar(10)|   NULL|
|WeekDayName_Short|    char(3)|   NULL|
|       DOWInMonth|    tinyint|   NULL|
|        DayOfYear|   smallint|   NULL|
|      WeekOfMonth|    tinyint|   NULL|
|       WeekOfYear|    tinyint|   NULL|
|            Month|    tinyint|   NULL|
|        MonthName|varchar(10)|   NULL|
|  MonthName_Short|    char(3)|   NULL|
|          Quarter|    tinyint|   NULL|
|      QuarterName| varchar(6)|   NULL|
|             Year|        int|   NULL|
|           MMYYYY|    char(6)|   NULL|
|        MonthYear|    char(7)|   NULL|
|        IsWeekend|    boolean|   NULL|
+-----------------+-----------+-------+
only showing top 20 rows


,DateKey,Date,Day,DaySuffix,Weekday,WeekDayName,WeekDayName_Short,DOWInMonth,DayOfYear,WeekOfMonth,...,IsWeekend,IsHoliday,HolidayName,SpecialDays,FirstDateofYear,LastDateofYear,FirstDateofMonth,LastDateofMonth,FirstDateofWeek,LastDateofWeek
0,20000101,2000-01-01,1,st,7,Saturday,SAT,1,1,1,...,True,True,New Years Day,None,2000-01-01,2000-12-31,2000-01-01,2000-01-31,1999-12-26,2000-01-01
1,20000102,2000-01-02,2,nd,1,Sunday,SUN,2,2,1,...,True,False,None,None,2000-01-01,2000-12-31,2000-01-01,2000-01-31,2000-01-02,2000-01-08


This code is used to populate the customers dimension and fetch the data from the table in MySQl. 

In [30]:
sql_dim_customers = f"SELECT * FROM {mysql_args['db_name']}.customer"
df_dim_customers = get_mysql_dataframe(spark, sql_dim_customers, **mysql_args)

df_dim_customers.toPandas().head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType,rowguid,ModifiedDate
0,1,1,AW00000001,S,b'^\xe9Z?}\xb8\xedJ\x95\xb4\xc3yz\xfc\xb7O',2004-10-13 11:15:07
1,2,1,AW00000002,S,b'W\xf6R\xe5\xaf\xa9}J\xa6E\xc4)\xd6\xe0$\x91',2004-10-13 11:15:07


In [31]:
# ----------------------------------------------------------------------------------
# Rename the 'CustomerID' column to 'customer_id'
# ----------------------------------------------------------------------------------
df_dim_customers = df_dim_customers.withColumnRenamed("CustomerID", "customer_id")

df_dim_customers.createOrReplaceTempView("customers")

sql_customers = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key
    FROM customers
"""
df_dim_customers = spark.sql(sql_customers)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows
# ----------------------------------------------------------------------------------
ordered_columns = ['customer_key', 'customer_id', 'AccountNumber', 
                   'CustomerType', 'TerritoryID', 'rowguid', 'ModifiedDate']

df_dim_customers = df_dim_customers.select(ordered_columns)

df_dim_customers.toPandas().head(2)

,customer_key,customer_id,AccountNumber,CustomerType,TerritoryID,rowguid,ModifiedDate
0,1,1,AW00000001,S,1,b'^\xe9Z?}\xb8\xedJ\x95\xb4\xc3yz\xfc\xb7O',2004-10-13 11:15:07
1,2,2,AW00000002,S,1,b'W\xf6R\xe5\xaf\xa9}J\xa6E\xc4)\xd6\xe0$\x91',2004-10-13 11:15:07


In [33]:
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

In [34]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_customers;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        customer_key|                 int|   NULL|
|         customer_id|                 int|   NULL|
|       AccountNumber|         varchar(10)|   NULL|
|        CustomerType|          varchar(1)|   NULL|
|         TerritoryID|                 int|   NULL|
|             rowguid|              binary|   NULL|
|        ModifiedDate|           timestamp|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|       dim_customers|       |
|        Created Time|Fri May 08 15:04:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 4.1.1|       |
|                Type|             MANAGED|       |
|           

,customer_key,customer_id,AccountNumber,CustomerType,TerritoryID,rowguid,ModifiedDate
0,1,1,AW00000001,S,1,b'^\xe9Z?}\xb8\xedJ\x95\xb4\xc3yz\xfc\xb7O',2004-10-13 11:15:07
1,2,2,AW00000002,S,1,b'W\xf6R\xe5\xaf\xa9}J\xa6E\xc4)\xd6\xe0$\x91',2004-10-13 11:15:07


Verify Dimension Table

In [35]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,dim_customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,dim_employee,False
3,adventureworks_dlh,dim_product,False
4,adventureworks_dlh,dim_productcategory,False
5,adventureworks_dlh,dim_salesterritory,False
6,,customers,True
7,,employees,True
8,,productcategory,True
9,,products,True


Use pyspark structured streaming to process fact data 

Verify the location of source data files on the file system. 

The sales order fact table data was split into 3 separate JSON files by year. 
The Spark AutoLoader reads one file at a time, processing each file as a 
separate streaming interval:

In [40]:
get_file_info(sales_orders_stream_dir)


,name,size,modification_time
0,sales_orders_2001.json,3274416,2026-05-08 19:18:57.520513058
1,sales_orders_2002.json,12332674,2026-05-08 19:18:58.025208473
2,sales_orders_2003.json,32363627,2026-05-08 19:18:59.393723965


This next section creates the bronze layer. This is reads the raw Data into a stream. 

In [41]:
df_sales_orders_bronze = (
    spark.readStream \
    .option("schemaLocation", sales_orders_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiLine", "true") \
    .json(sales_orders_stream_dir)
)

df_sales_orders_bronze.isStreaming

True

This writes the streaming data to a parquet file. 

In [42]:
sales_orders_checkpoint_bronze = os.path.join(sales_orders_output_bronze, '_checkpoint')

sales_orders_bronze_query = (
    df_sales_orders_bronze
    # Add Current Timestamp and Input Filename columns for Traceability
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("sales_orders_bronze")
    .trigger(availableNow = True) \
    .option("checkpointLocation", sales_orders_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(sales_orders_output_bronze)
)

This tests by monitoring the query. 

In [43]:
print(f"Query ID: {sales_orders_bronze_query.id}")
print(f"Query Name: {sales_orders_bronze_query.name}")
print(f"Query Status: {sales_orders_bronze_query.status}")

Query ID: 61545eee-f444-46d1-8180-d4d3f32d82c6
Query Name: sales_orders_bronze
Query Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [44]:
sales_orders_bronze_query.awaitTermination()

This code is used to create the silver layer by join Streaming Fact Data with Reference Dimensions. 

In [61]:
df_dim_customers_j = df_dim_customers.withColumnRenamed("customer_id", "dim_customer_id")
df_dim_employees_j = df_dim_employees.withColumnRenamed("EmployeeID", "dim_employee_id")
df_dim_products_j = df_dim_product.withColumnRenamed("product_id", "dim_product_id")
df_dim_salesterritory_j = df_dim_salesterritory.withColumnRenamed("territory_id", "dim_territory_id")

df_dim_order_date_j = df_dim_date.select(
    col("DateKey").alias("order_date_key"),
    col("Date").alias("dim_order_date"),
    col("Year"),
    col("Quarter"),
    col("Month"),
    col("MonthName")
)

df_dim_due_date_j = df_dim_date.select(
    col("DateKey").alias("due_date_key"),
    col("Date").alias("dim_due_date")
)

df_dim_ship_date_j = df_dim_date.select(
    col("DateKey").alias("ship_date_key"),
    col("Date").alias("dim_ship_date")
)

This code joins the streaming with batch data to define the silver query. 

In [62]:

df_sales_orders_silver = spark.readStream.format("parquet").load(sales_orders_output_bronze) \
    .join(df_dim_customers_j, df_dim_customers_j.dim_customer_id == col("customer_id").cast(IntegerType()), "inner") \
    .join(df_dim_employees_j, df_dim_employees_j.dim_employee_id == col("salesperson_id").cast(IntegerType()), "left_outer") \
    .join(df_dim_products_j, df_dim_products_j.dim_product_id == col("product_id").cast(IntegerType()), "inner") \
    .join(df_dim_salesterritory_j, df_dim_salesterritory_j.dim_territory_id == col("territory_id").cast(IntegerType()), "left_outer") \
    .join(df_dim_order_date_j, df_dim_order_date_j.dim_order_date.cast(DateType()) == col("order_date").cast(DateType()), "inner") \
    .join(df_dim_due_date_j, df_dim_due_date_j.dim_due_date.cast(DateType()) == col("due_date").cast(DateType()), "left_outer") \
    .join(df_dim_ship_date_j, df_dim_ship_date_j.dim_ship_date.cast(DateType()) == col("ship_date").cast(DateType()), "left_outer") \
    .select(
        col("sales_order_id").cast(LongType()),
        col("sales_order_detail_id").cast(LongType()),
        df_dim_customers_j.customer_key.cast(LongType()),
        df_dim_employees_j.employee_key.cast(LongType()),
        df_dim_products_j.product_key.cast(LongType()),
        df_dim_salesterritory_j.territory_key.cast(IntegerType()),
        df_dim_order_date_j.order_date_key.cast(LongType()),
        df_dim_due_date_j.due_date_key.cast(LongType()),
        df_dim_ship_date_j.ship_date_key.cast(LongType()),
        col("order_qty"),
        col("unit_price"),
        col("unit_price_discount"),
        col("line_total"),
        col("sub_total"),
        col("tax_amt"),
        col("freight"),
        col("total_due"),
        col("status"),
        col("online_order_flag")
    )

In [63]:
df_sales_orders_silver.isStreaming

True

In [64]:
df_sales_orders_silver.printSchema()

root
 |-- sales_order_id: long (nullable = true)
 |-- sales_order_detail_id: long (nullable = true)
 |-- customer_key: long (nullable = false)
 |-- employee_key: long (nullable = true)
 |-- product_key: long (nullable = false)
 |-- territory_key: integer (nullable = true)
 |-- order_date_key: long (nullable = true)
 |-- due_date_key: long (nullable = true)
 |-- ship_date_key: long (nullable = true)
 |-- order_qty: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- unit_price_discount: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- sub_total: double (nullable = true)
 |-- tax_amt: double (nullable = true)
 |-- freight: double (nullable = true)
 |-- total_due: double (nullable = true)
 |-- status: long (nullable = true)
 |-- online_order_flag: string (nullable = true)



This code writes the tranformed data into the data lakehouse. 

In [65]:
orders_checkpoint_silver = os.path.join(sales_orders_output_silver, '_checkpoint')

sales_orders_silver_query = (
    df_sales_orders_silver.writeStream
    .format("parquet")
    .outputMode("append")
    .queryName("sales_orders_silver")
    .trigger(availableNow=True)
    .option("checkpointLocation", orders_checkpoint_silver)
    .option("compression", "snappy")
    .start(sales_orders_output_silver)
)

Verify the streaming data's status. 

In [66]:
print(f"Query ID: {sales_orders_silver_query.id}")
print(f"Query Name: {sales_orders_silver_query.name}")
print(f"Query Status: {sales_orders_silver_query.status}")

Query ID: c0ea7d1f-d3f3-4ad5-83f1-f99eb8d82e7a
Query Name: sales_orders_silver
Query Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [67]:
sales_orders_silver_query.awaitTermination()

This code renames conflicting columns in each dimension table and selects only the necessary columns from the date dimension to prepare them for joining with the streaming sales fact data.

In [68]:
df_fact_sales_by_category_gold = (
    spark.readStream.format("parquet").load(sales_orders_output_silver)
    .join(df_dim_products_j, "product_key")
    .join(df_dim_order_date_j, "order_date_key")
    .groupBy("ProductCategory", "Year")
    .agg(
        sum("line_total").alias("Total Revenue"),
        sum("order_qty").alias("Total Units Sold"),
        count("sales_order_id").alias("Total Orders")
    )
    .orderBy("Year", desc("Total Revenue"))
)

In [69]:
df_fact_sales_by_category_gold.printSchema()

root
 |-- ProductCategory: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Total Revenue: double (nullable = true)
 |-- Total Units Sold: long (nullable = true)
 |-- Total Orders: long (nullable = false)



Write the streaming data into memory in complete mode. 

In [71]:
sales_orders_gold_query_1 = (
    df_fact_sales_by_category_gold.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("fact_sales_by_category")
    .start()
)

In [73]:
wait_until_stream_is_ready(sales_orders_gold_query_1, 1)

The stream has processed 4 batchs


Queries the gold data from in-memory storage. 

In [74]:
df_fact_sales_by_category = spark.sql("SELECT * FROM fact_sales_by_category")
df_fact_sales_by_category.printSchema()

root
 |-- ProductCategory: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Total Revenue: double (nullable = true)
 |-- Total Units Sold: long (nullable = true)
 |-- Total Orders: long (nullable = false)



In [ ]:
This code creates the final selection by pulling the wanted columns. 

In [79]:
df_fact_sales_by_category_gold_final = (
    df_fact_sales_by_category
    .select(
        col("Year").cast(IntegerType()),
        col("ProductCategory").alias("Product Category"),
        round(col("Total Revenue"), 2).alias("Total Revenue"),
        col("Total Units Sold").cast(IntegerType()),
        col("Total Orders").cast(IntegerType())
    )
    .orderBy("Year", desc("Total Revenue"))
)

This data loads the final results into a new table and display the total sales revenue, units sold, and order count broken down by product category and year

In [80]:
df_fact_sales_by_category_gold_final.write.saveAsTable(f"{dest_database}.fact_sales_by_category", mode="overwrite")
spark.sql(f"SELECT * FROM {dest_database}.fact_sales_by_category").toPandas()

,Year,Product Category,Total Revenue,Total Units Sold,Total Orders
0,2002,Clothing,485587.15,16927,3482
1,2002,Accessories,92735.35,5207,1186
2,2003,Components,5485514.83,24118,8868
3,2003,Clothing,1011984.50,35377,9349
4,2001,Bikes,10661722.28,7139,3356
5,2001,Components,615474.98,1574,835
6,2003,Accessories,590257.59,28162,17377
7,2002,Components,3610092.47,13876,5031
8,2001,Accessories,20235.36,1003,346
9,2002,Bikes,26486358.20,24908,9654


This statement shows the total revenue, units sold, and orders broken down by product subcategory. 

In [92]:
spark.sql(f"""
    SELECT
        p.ProductCategory                                       AS `Product Category`,
        p.ProductSubcategory                                    AS `Subcategory`,
        format_number(SUM(f.`Total Revenue`), 2)               AS `Total Revenue`,
        SUM(f.`Total Units Sold`)                              AS `Total Units Sold`,
        SUM(f.`Total Orders`)                                  AS `Total Orders`
    FROM {dest_database}.fact_sales_by_category AS f
    INNER JOIN {dest_database}.dim_product AS p
        ON f.`Product Category` = p.ProductCategory
    GROUP BY p.ProductCategory, p.ProductSubcategory
    ORDER BY SUM(f.`Total Revenue`) DESC
""").toPandas()

,Product Category,Subcategory,Total Revenue,Total Units Sold,Total Orders
0,Bikes,Road Bikes,"3,099,068,510.96",2970827,1232079
1,Bikes,Mountain Bikes,"2,306,283,543.04",2210848,916896
2,Bikes,Touring Bikes,"1,585,569,935.84",1519958,630366
3,Components,Road Frames,"320,465,715.24",1305744,486222
4,Components,Mountain Frames,"271,910,303.84",1107904,412552
5,Components,Touring Frames,"174,799,481.04",712224,265212
6,Components,Wheels,"135,955,151.92",553952,206276
7,Components,Saddles,"87,399,740.52",356112,132606
8,Components,Handlebars,"77,688,658.24",316544,117872
9,Components,Pedals,"67,977,575.96",276976,103138


This code shows the total revenue by quarter across all years. 


In [93]:
spark.sql(f"""
    SELECT
        d.Year                                                      AS `Year`,
        d.Quarter                                                   AS `Quarter`,
        d.QuarterName                                               AS `Quarter Name`,
        format_number(SUM(f.`Total Revenue`), 2)                   AS `Total Revenue`,
        SUM(f.`Total Units Sold`)                                  AS `Total Units Sold`,
        SUM(f.`Total Orders`)                                      AS `Total Orders`
    FROM {dest_database}.fact_sales_by_category AS f
    INNER JOIN {dest_database}.dim_date AS d
        ON f.Year = d.Year
    GROUP BY d.Year, d.Quarter, d.QuarterName
    ORDER BY d.Year, d.Quarter ASC
""").toPandas()

,Year,Quarter,Quarter Name,Total Revenue,Total Units Sold,Total Orders
0,2001,1,First,"1,019,862,806.40",1066320,463590
1,2001,2,Second,"1,031,194,615.36",1078168,468741
2,2001,3,Third,"1,042,526,424.32",1090016,473892
3,2001,4,Fourth,"1,042,526,424.32",1090016,473892
4,2002,1,First,"2,760,729,585.30",5482620,1741770
5,2002,2,Second,"2,791,404,358.47",5543538,1761123
6,2002,3,Third,"2,822,079,131.64",5604456,1780476
7,2002,4,Fourth,"2,822,079,131.64",5604456,1780476
8,2003,1,First,"3,780,993,344.40",11222910,4611330
9,2003,2,Second,"3,823,004,381.56",11347609,4662567
